In [149]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import collections
import re
seed = 0
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

with open('kab.txt',mode='r') as file:
    rows = file.readlines()

english = []
kabyle = []
for r in rows:
    parts = r.split('\t')
    tokens_english = re.findall(r"\w+|[^\w\s]", parts[0])
    tokens_kabyle = re.findall(r"\w+|[^\w\s]", parts[1])
    for i in range(0,len(tokens_english)):
        tokens_english[i] = tokens_english[i].lower()
    for i in range(0,len(tokens_kabyle)):
        tokens_kabyle[i] = tokens_kabyle[i].lower()
    english.append(tokens_english)
    kabyle.append(tokens_kabyle)
    

UNK_TOKEN = "<unk>"
PAD_TOKEN = "<pad>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"

UNK_IDX = 0
PAD_IDX = 1
SOS_IDX = 2
EOS_IDX = 3

class Vocabulary:
    def __init__(self):
        self.min_frequency = 3
        self.idx2char_english = {
            UNK_IDX:UNK_TOKEN,
            PAD_IDX:PAD_TOKEN,
            SOS_IDX:SOS_TOKEN,
            EOS_IDX:EOS_TOKEN
        }
        self.char2idx_english = {v: k for k, v in self.idx2char_english.items()}
        self.idx2char_kabyle = {
            UNK_IDX:UNK_TOKEN,
            PAD_IDX:PAD_TOKEN,
            SOS_IDX:SOS_TOKEN,
            EOS_IDX:EOS_TOKEN
        }
        self.char2idx_kabyle = {v: k for k, v in self.idx2char_kabyle.items()}
    def create_vocabulary_kabyle(self,tokenized_texts):
        counter = collections.Counter()
        for t in tokenized_texts:
            counter.update(t)
        idx = len(self.idx2char_kabyle)
        for value,count in counter.items():
            if count >= self.min_frequency and value not in self.char2idx_kabyle:
                self.char2idx_kabyle[value] = idx
                self.idx2char_kabyle[idx] = value
                idx += 1
    def create_vocabulary_english(self,tokenized_texts):
        counter = collections.Counter()
        for t in tokenized_texts:
            counter.update(t)
        idx = len(self.idx2char_english)
        for value,count in counter.items():
            if count >= self.min_frequency and value not in self.char2idx_english:
                self.char2idx_english[value] = idx
                self.idx2char_english[idx] = value
                idx += 1
    def transformation_kabyle(self, tokens, max_length=1000):
        result = []
        for i in tokens:
            truncated = i[:max_length]
            ids = [SOS_IDX] + [self.char2idx_kabyle.get(i, UNK_IDX) for i in truncated]+ [EOS_IDX]
            ids = ids + [PAD_IDX] * (65-len(ids))
            result.append(ids)
        return result
    def transformation_english(self, tokens, max_length=1000):
        result = []
        for i in tokens:
            truncated = i[:max_length]
            ids = [SOS_IDX] + [self.char2idx_english.get(i, UNK_IDX) for i in truncated]+ [EOS_IDX]
            ids = ids + [PAD_IDX] * (50-len(ids))
            result.append(ids)
        return result

In [150]:
voc = Vocabulary()
voc.create_vocabulary_english(english)
voc.create_vocabulary_kabyle(kabyle)

tokenized_english = voc.transformation_english(english)
tokenized_kabyle = voc.transformation_kabyle(kabyle)

In [151]:
from torch.utils.data import Dataset,DataLoader

class DataLoadingDataset(Dataset):
    def __init__(self,eng,kab):
        self.eng = eng
        self.kab = kab
    def __len__(self):
        return(len(self.kab))
    def __getitem__(self,idx):
        kab = self.kab[idx]
        eng = self.eng[idx]
        return {'eng':eng,'kab':kab}
    
def get_dataloader():
    def collate_fn(batch):
        batch_ids_english = [i['eng'] for i in batch]
        batch_ids_kabyle = [i['kab'] for i in batch]
        batch_ids_english = nn.utils.rnn.pad_sequence(torch.tensor(batch_ids_english,dtype=torch.int),padding_value=1)
        batch_ids_kabyle = nn.utils.rnn.pad_sequence(torch.tensor(batch_ids_kabyle,dtype=torch.int),padding_value=1)
        batch = {
            "eng": torch.tensor(batch_ids_english,dtype=torch.int),
            "kab": torch.tensor(batch_ids_kabyle,dtype=torch.int),
        }
        return batch
    return collate_fn

def get_data_loader():
    collate_fn = get_dataloader()
    data_loader = torch.utils.data.DataLoader(
        dataset=DataLoadingDataset(tokenized_english,tokenized_kabyle),
        batch_size=64,
        collate_fn=collate_fn,
        shuffle=True,
    )
    return data_loader
    
data = get_data_loader()

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(3876,256,[0,1])
        self.rnn = nn.LSTM(256,512,4,dropout=0.2)
        
